# Baseline Evaluation

Runs physics and learned baselines against the IQ L=50 dataset.
All baselines are compared on the same held-out test split.

**Metrics** (accumulated globally across the whole test set, not averaged per-batch;
see `Baselines/metrics.py` for definitions):
- MSLE (mean squared log1p error), R² in raw I(q) space, R² in log1p(I) space, CPU time per atom -- summary bar chart
- Per-q R² and per-q mean percent error -- does performance hold up at high q?
- Kratky-style log1p(I(q)) vs q overlay (true vs. predicted) -- curve shape, not just scale
- Residual histogram and error-vs-atom-count -- bias and size-scaling diagnostics

A single pooled R² (in either space) is dominated by molecule-to-molecule scale
variance, so no one number here should be read as "the" score -- see `Baselines/metrics.py`
module docstring.

**Checkpointing / resume**: results are pushed to Drive (via the same `RCLONE_CONF`
Kaggle secret used by `kaggle_train.ipynb`) after every baseline finishes. If the
session dies mid-run, just rerun the notebook top to bottom -- already-completed
baselines are detected and skipped automatically.

In [ ]:
import os, subprocess, sys

NOTEBOOK_NAME = "your-kaggle-notebook"   # set to your Kaggle notebook slug
assert NOTEBOOK_NAME != "your-kaggle-notebook", "Set NOTEBOOK_NAME first."

HDF5_PATH = f"/kaggle/input/datasets/noso0s0n/iql50/I(q)L50.h5"
REPO      = f"/kaggle/working/{NOTEBOOK_NAME}"
DB_NAME   = f"/kaggle/input/datasets/noso0s0n/iql50/iq_train_set-ENCODING.sqlite3"   # mounted from the Kaggle dataset
N_BUCKETS = 10   # number of atom-size buckets to evaluate, randomly sampled (< 57 for speed)

# install deps
%pip install -q xraydb beartype jaxtyping hdf5plugin h5py "scikit-learn>=1.3"   # >=1.3 for sklearn.cluster.HDBSCAN
!curl -fsSL https://rclone.org/install.sh | sudo bash

# plots (Baselines/metrics.py) render text through real LaTeX (xelatex) with
# JuliaMono as the font -- classic latex+dvipng (matplotlib's usetex default)
# can't load an arbitrary system font, only xelatex/lualatex + fontspec can.
!sudo apt-get update -q && sudo apt-get install -y -q texlive-xetex texlive-latex-recommended texlive-fonts-recommended
!mkdir -p ~/.fonts && curl -fsSL https://github.com/cormullion/juliamono/releases/latest/download/JuliaMono-ttf.tar.gz \
    | tar -xz -C ~/.fonts && fc-cache -f ~/.fonts

# clone repo
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "https://github.com/noshou/APS360.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull"], check=True)

sys.path.insert(0, REPO)

In [ ]:
# ── rclone / Google Drive checkpointing ── run once per session ─────────────
# Reuses the same RCLONE_CONF Kaggle secret as kaggle_train.ipynb (it's just
# remote storage credentials, not training-specific). Checkpoints under a
# separate baselines_ckpts/ prefix so this never collides with training
# checkpoints on the same Drive.
import base64, json
from kaggle_secrets import UserSecretsClient
from Baselines.metrics import EvalResult

conf_path = "/kaggle/working/rclone.conf"
with open(conf_path, "w") as f:
    f.write(base64.b64decode(UserSecretsClient().get_secret("RCLONE_CONF")).decode())
os.environ["RCLONE_CONFIG"] = conf_path

remotes = subprocess.run(["rclone", "listremotes"], capture_output=True, text=True).stdout.strip().split("\n")
remote  = remotes[0] if remotes and remotes[0] else ""
assert remote.endswith(":"), (
    f"No rclone remote found (rclone listremotes -> {remotes!r}). Check the RCLONE_CONF secret."
)
REMOTE_NAME = remote + f"{NOTEBOOK_NAME}/baselines_ckpts/"

out = subprocess.run(["rclone", "mkdir", REMOTE_NAME], capture_output=True, text=True)
print("remote drive ──>", REMOTE_NAME, "(ok)" if out.returncode == 0 else f"ERROR: {out.stderr.strip()}")

_CKPT_LOCAL = "/kaggle/working/baselines_results.json"

_CKPT_REMOTE_NAME = os.path.basename(_CKPT_LOCAL)   # "baselines_results.json" -- must match on
                                                      # both ends, or save/load silently target
                                                      # different files and nothing ever resumes

def load_checkpoint() -> dict:
    """Pull the checkpoint from Drive if it exists and return completed baselines.

    Returns name -> EvalResult (the full result, including per-q arrays needed
    for plotting -- not just the summary numbers), so a resumed baseline still
    has plot data and R²(raw) available this session. Returns an empty dict on
    a fresh run (nothing to resume) or if the Drive pull failed; the printed
    message distinguishes the two. Baselines whose name is already a key here
    get skipped by the evaluation loops below.
    """
    out = subprocess.run(
        ["rclone", "copy", f"{REMOTE_NAME}{_CKPT_REMOTE_NAME}", "/kaggle/working/"],
        capture_output=True, text=True,
    )
    if not os.path.exists(_CKPT_LOCAL):
        if out.returncode != 0:
            print(f"No existing checkpoint on Drive (or pull failed): {out.stderr.strip()}")
        return {}
    with open(_CKPT_LOCAL) as f:
        raw = json.load(f)
    data = {name: EvalResult.from_json(v) for name, v in raw.items()}
    print(f"Resumed {len(data)} completed baseline(s) from checkpoint: {list(data.keys())}")
    return data

def save_checkpoint(results: dict) -> None:
    """Write the checkpoint locally and push it to Drive. Call after every baseline.

    Verifies the push with `rclone lsf` rather than trusting `copy`'s exit code --
    rclone can return 0 while nothing actually lands on Drive (stale/expired
    OAuth token, wrong root_folder_id, Shared-Drive permissions), which is the
    silent-failure mode that makes a checkpoint look saved when it never
    reached Drive at all.
    """
    with open(_CKPT_LOCAL, "w") as f:
        json.dump({name: r.to_json() for name, r in results.items()}, f, indent=2)
    out = subprocess.run(
        ["rclone", "copy", _CKPT_LOCAL, REMOTE_NAME], capture_output=True, text=True
    )
    if out.returncode != 0:
        print(f"WARNING: rclone push of checkpoint failed: {out.stderr.strip()}")
        return
    verify = subprocess.run(
        ["rclone", "lsf", f"{REMOTE_NAME}{_CKPT_REMOTE_NAME}"], capture_output=True, text=True
    )
    if verify.returncode != 0 or not verify.stdout.strip():
        print(
            f"WARNING: rclone copy exited 0 but the checkpoint isn't visible on Drive "
            f"afterward ({verify.stderr.strip() or 'empty listing'}) -- this session's "
            f"checkpoint is NOT actually saved to Drive; check RCLONE_CONF / Drive permissions"
        )
    else:
        print(f"checkpoint pushed to Drive ({len(results)} baseline(s) saved)")

In [ ]:
import h5py, hdf5plugin
from Preprocess.encode import Encoding
from ScatterNet.utils.config import DEFAULT_BUCKETS

print("Loading encoding DB (iq_train_set-ENCODING.sqlite3, mounted from Kaggle dataset)...")
enc = Encoding(DB_NAME, HDF5_PATH)
print(f"  {enc.count():,} molecules  |  max atoms: {enc._max}")

with h5py.File(HDF5_PATH, "r") as f:
    q_grid = f["q_grid"][()]
    energy = float(f.attrs.get("energy", 10000.0))

import torch
q_grid = torch.from_numpy(q_grid).float()
print(f"  q_grid: {len(q_grid)} points  |  energy: {energy} eV")

In [ ]:
import random
import time
from ScatterNet.batching import Batcher, Batch
from torch.utils.data import DataLoader

BUCKET_SAMPLE_SEED = 3092983   # deterministic; change to sample a different subset of buckets
LOADER_WORKERS = 4              # parallel HDF5 reads while materializing each split once

def _first(x):
    return x[0]

def _materialize(dataset, name, num_workers=LOADER_WORKERS):
    """Read every batch out of `dataset` once via a parallel DataLoader and cache
    it as a plain list. Every baseline's .fit()/evaluate() call then iterates this
    list directly instead of re-triggering per-molecule HDF5 reads from scratch
    on every single pass -- with 5 fits + 8 evaluates sharing train/test data,
    that turns ~13 full HDF5 passes into 2.
    """
    loader = DataLoader(dataset, batch_size=1, collate_fn=_first, num_workers=num_workers)
    out, t0 = [], time.time()
    for i, batch in enumerate(loader):
        out.append(batch)
        print(f"\r  materializing {name}: {i+1}/{len(dataset)}  ({time.time()-t0:.0f}s elapsed)",
                end="", flush=True)
    print()
    return out

eval_buckets = sorted(
    random.Random(BUCKET_SAMPLE_SEED).sample(DEFAULT_BUCKETS, min(N_BUCKETS, len(DEFAULT_BUCKETS)))
)

batcher = Batcher(
    hdf5_db        = HDF5_PATH,
    enc            = enc,
    batches        = eval_buckets,
    seed           = 42,
    atom_size_ceil = 6046,
)
_, _, test_set = batcher.get_sets()

test_loader = _materialize(test_set, "test set")
print(f"Test batches: {len(test_loader)}")

In [ ]:
from Baselines.metrics import evaluate as _evaluate

def evaluate(baseline, loader, name):
    """Evaluate a baseline, print its metrics, and return the full EvalResult.

    See Baselines/metrics.py for the full metric definitions (MSLE, R²
    raw/log1p, per-q breakdowns, etc). The full result (including the per-q
    arrays needed for plotting) is what gets checkpointed now, so a baseline
    resumed from checkpoint in a later session still plots and reports
    R²(raw) correctly.
    """
    result = _evaluate(baseline, loader, q_grid, name)
    print(f"{name:<30s}  MSLE={result.msle:.4f}  R²(raw)={result.r2_raw:.4f}  "
          f"R²(log1p)={result.r2_log1p:.4f}  {result.us_per_atom:.2f} μs/atom")
    return result

In [ ]:
sys.path.insert(0, f"{REPO}/Baselines/physics-benchmarks")
sys.path.insert(0, f"{REPO}/Baselines/learned-benchmarks")

from rg import RgBaseline, GuinierPorodBaseline
from atom_count import AtomCountBaseline
from pair_peak import BinnedDebyeBaseline

print("=== Physics Baselines ===")
results = load_checkpoint()

# fit on train set where needed -- materialized once, reused by every .fit() below
# and by the learned baselines further down, instead of re-reading HDF5 per call
train_set, _, _ = batcher.get_sets()
train_loader = _materialize(train_set, "train set")

# factories, not baseline instances: construction/fit is deferred until we know
# a baseline isn't already in `results`, so a completed baseline's .fit() never
# re-runs on resume (also fixes the earlier issue where the list literal ran
# every .fit() eagerly before any evaluate() call could print)
physics_baselines = [
    ("Guinier (Rg)",          lambda: RgBaseline(q_grid, energy)),
    ("Guinier-Porod",         lambda: GuinierPorodBaseline(q_grid, energy)),
    ("Atom Count",            lambda: AtomCountBaseline().fit(train_loader)),
    ("Binned Debye",          lambda: BinnedDebyeBaseline(q_grid, energy)),
]

for name, make_baseline in physics_baselines:
    if name in results:
        print(f"{name:<30s}  (skipped, resumed from checkpoint)")
        continue
    results[name] = evaluate(make_baseline(), test_loader, name)
    save_checkpoint(results)

In [ ]:
import torch
from torch_mlp import TorchMlp   # Baselines/learned-benchmarks, added to sys.path above

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
from linsvm import Linsvm
from nearest_neighbour import NNBaseline
from hdbscan import Hdbscan

print("=== Learned Baselines ===")

learned_baselines = [
    ("MLP",                lambda: TorchMlp().fit(train_loader)),
    ("Linear SVM",         lambda: Linsvm().fit(train_loader)),
    ("Nearest Neighbour",  lambda: NNBaseline(q_grid, energy).fit(train_loader)),
    ("HDBSCAN",            lambda: Hdbscan(q_grid, energy)),   # analytical, no .fit() needed
]

for name, make_baseline in learned_baselines:
    if name in results:
        print(f"{name:<30s}  (skipped, resumed from checkpoint)")
        continue
    results[name] = evaluate(make_baseline(), test_loader, name)
    save_checkpoint(results)

In [ ]:
_known_names = {n for n, _ in physics_baselines} | {n for n, _ in learned_baselines}
_stale = sorted(n for n in results if n not in _known_names)
if _stale:
    print(f"Dropping stale checkpoint entries no longer in the baseline list: {_stale}")
    for n in _stale:
        del results[n]
    save_checkpoint(results)

print("\n=== Summary ===")
print(f"{'Baseline':<30s}  {'MSLE':>8s}  {'R²(raw)':>10s}  {'R²(log1p)':>12s}  {'μs/atom':>10s}")
print("-" * 78)
for name, r in sorted(results.items(), key=lambda x: x[1].msle):
    print(f"{name:<30s}  {r.msle:>8.4f}  {r.r2_raw:>10.4f}  {r.r2_log1p:>12.4f}  {r.us_per_atom:>10.2f}")

In [ ]:
from Baselines.metrics import run_all_plots

written = run_all_plots(list(results.values()), q_grid, "/kaggle/working/baseline_plots")
print("wrote:", *written, sep="\n  ")

In [ ]:
# per-q R², per-q percent error, Kratky overlay, residual histogram, and
# error-vs-atom-count are all written by run_all_plots() in the cell above --
# see Baselines/metrics.py for what each one shows and why.